# Comparaison de modèles Ultralytics — YOLO26n, YOLO11n, YOLO12n

Notebook **autonome** : il construit les folds depuis le CSV de split, puis compare les modèles en CV (folds 0-4). **fold5 n'est pas utilisé ici.**

Ordre : install → Drive → **config données** → **construction des folds (depuis le CSV)** → config comparaison → W&B → vérif → comparaison CV. Sorties dans `puceron_model_comparaison/data`.

In [2]:
# --- Installation + imports ---
!pip install -q ultralytics wandb
import os, time, shutil, random, yaml, zipfile
import numpy as np, pandas as pd
from pathlib import Path
from ultralytics import YOLO

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 10.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [3]:
# --- Connexion Drive ---
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# --- CONFIG DONNÉES (source des tuiles + variante labels_cell_20) ---
BASE_DIR  = Path("/content/drive/MyDrive/Emma/puceron_model_2026/puceron_model_E2026_3/data/tuile_viz02_640_128")
SPLIT_DIR = BASE_DIR / "split"
CELL_DIR  = Path("/content/drive/MyDrive/Emma/puceron_model_2026/puceron_model_E2026_2/data/cell/tuile_viz02_640_128_cell")

def V(csv, labels, extra_train=None):
    labels = Path(labels)
    if not labels.is_absolute():
        labels = BASE_DIR / labels
    variant = {"csv": Path(csv), "labels_dir": labels}
    if extra_train is not None:
        variant["extra_train"] = {"csv": Path(extra_train["csv"]),
                                  "labels_dir": Path(extra_train["labels_dir"]),
                                  "images_dir": Path(extra_train["images_dir"])}
    return variant

EXPERIMENT_NAME = "yolo_neg1"
VARIANTS = {
    "labels_cell_20": V(
        SPLIT_DIR / "split_assignments_all_background.csv", "labels_visible_20",
        extra_train={"csv":        CELL_DIR / "split" / "split_assignments.csv",
                     "labels_dir": CELL_DIR / "labels_cell_20",
                     "images_dir": CELL_DIR / "images_lookmatched3"}),
}
BASELINE_VARIANT = "labels_cell_20"
SEARCH_VARIANT   = BASELINE_VARIANT
USE_BACKGROUND   = True
MAIN_IMAGES_DIR  = BASE_DIR / "images"
BG_DIR = Path("/content/drive/MyDrive/Emma/puceron_model_2026/puceron_model_E2026_2/data/images_complete")
OUTPUT_ROOT = Path(f"/content/kfold_yolo/{EXPERIMENT_NAME}")
YAML_DIR    = Path(f"/content/yolo_yaml/{EXPERIMENT_NAME}"); YAML_DIR.mkdir(parents=True, exist_ok=True)
N_CV_FOLDS      = 5
TEST_FOLD_VALUE = 5
CLASS_NAMES     = {0: "Apterous_aphid", 1: "Alate_aphid"}
print("Config données OK | variante :", SEARCH_VARIANT, "| CV folds :", N_CV_FOLDS)

Config données OK | variante : labels_cell_20 | CV folds : 5


In [5]:
# ============================================================================
# CONSTRUCTION DES FOLDS À PARTIR DU CSV DE SPLIT (à exécuter APRÈS la config)
# Lit split_assignments_all_background.csv : chaque ligne -> son fold (colonne split).
#   label_file vide  -> tuile de FOND (image seule, pas de label)
#   label_file rempli -> PUCERON (image + label depuis labels_dir)
# Ajoute aussi le renfort "cell" (_extra_train, ajouté au train uniquement).
# ============================================================================
import zipfile, pandas as pd
from pathlib import Path

STAGE_LOCAL = Path("/content/stage_compare"); STAGE_LOCAL.mkdir(parents=True, exist_ok=True)
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def stage_dir(src):
    """Retourne un dossier LOCAL contenant les images de src.
       - si le dossier Drive est peuplé -> on l'utilise directement ;
       - sinon si <src>.zip existe -> on l'extrait UNE fois en local."""
    src = Path(src)
    if src.exists() and any(True for _ in src.iterdir()):
        return src
    zip_src = f"{src.parent}/tuile_viz02_640_128_background.zip"
    print(zip_src)
    local = STAGE_LOCAL / src.name
    if (local / ".done").exists():
        return local
    if zip_src.exists():
        local.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_src) as zf:
            zf.extractall(local)
        (local / ".done").write_text("ok")
        return local
    raise FileNotFoundError(f"Ni dossier peuplé ni archive .zip pour {src}")

def build_image_index(dirs):
    """Index {nom_fichier -> chemin} en fusionnant plusieurs dossiers sources."""
    idx = {}
    for d in dirs:
        base = stage_dir(d)
        for p in Path(base).rglob("*"):
            if p.is_file() and p.suffix.lower() in IMG_EXTS:
                idx.setdefault(p.name, p)      # premier trouvé gagne
    return idx

def _normalize_fold(v):
    """'fold3' -> 3 (tolère aussi un entier)."""
    return int(str(v).strip().lower().replace("fold", ""))

def _link(src, dst):
    if not dst.exists():
        try:
            dst.symlink_to(src)
        except FileExistsError:
            pass

def build_folds_from_csv():
    variant   = VARIANTS[BASELINE_VARIANT]
    df        = pd.read_csv(variant["csv"])
    labels_dir = Path(variant["labels_dir"])
    root      = OUTPUT_ROOT / SEARCH_VARIANT

    print("Indexation des images sources (pucerons + fonds)...")
    img_idx = build_image_index([MAIN_IMAGES_DIR, BG_DIR])
    print(f"  {len(img_idx)} images indexées.")

    for f in range(N_CV_FOLDS + 1):            # folds 0..5 (5 = test)
        (root / f"fold_{f}" / "images").mkdir(parents=True, exist_ok=True)
        (root / f"fold_{f}" / "labels").mkdir(parents=True, exist_ok=True)

    ok = miss = 0
    for name, lf, split in zip(df["image"], df["label_file"], df["split"]):
        fold = _normalize_fold(split)
        src_img = img_idx.get(name)
        if src_img is None:
            miss += 1; continue
        _link(src_img, root / f"fold_{fold}" / "images" / name)
        if isinstance(lf, str) and lf.strip():             # puceron -> label
            lsrc = labels_dir / lf
            if lsrc.exists():
                _link(lsrc, root / f"fold_{fold}" / "labels" / (Path(name).stem + ".txt"))
        ok += 1
    print(f"CSV principal : {ok} tuiles placées, {miss} images introuvables")

    # --- Renfort "cell" -> _extra_train (ajouté au TRAIN uniquement) ---
    ex = variant.get("extra_train")
    if ex:
        (root / "_extra_train" / "images").mkdir(parents=True, exist_ok=True)
        (root / "_extra_train" / "labels").mkdir(parents=True, exist_ok=True)
        ex_df   = pd.read_csv(ex["csv"])
        ex_imgs = build_image_index([ex["images_dir"]])
        ex_lbls = Path(ex["labels_dir"])
        n_ex = 0
        for name in ex_df["image"]:
            src_img = ex_imgs.get(name)
            if src_img is None:
                continue
            _link(src_img, root / "_extra_train" / "images" / name)
            lsrc = ex_lbls / (Path(name).stem + ".txt")
            if lsrc.exists():
                _link(lsrc, root / "_extra_train" / "labels" / (Path(name).stem + ".txt"))
            n_ex += 1
        print(f"_extra_train (cell) : {n_ex} images ajoutées (train uniquement)")

    return root

build_folds_from_csv()

# --- Vérification : compter pucerons/fonds réellement placés par fold ---
def _has_annotations(lbl):
    lbl = Path(lbl)
    return lbl.exists() and any(l.strip() for l in lbl.read_text().splitlines())

root = OUTPUT_ROOT / SEARCH_VARIANT
print("\nVérification (doit correspondre au CSV) :")
for f in range(N_CV_FOLDS + 1):
    imgs = list((root / f"fold_{f}" / "images").glob("*"))
    npuc = sum(1 for im in imgs if _has_annotations(root / f"fold_{f}" / "labels" / f"{im.stem}.txt"))
    print(f"  fold {f} : {len(imgs)} images ({npuc} pucerons, {len(imgs)-npuc} fonds)")


Indexation des images sources (pucerons + fonds)...
  2535 images indexées.
CSV principal : 1767 tuiles placées, 69849 images introuvables
_extra_train (cell) : 532 images ajoutées (train uniquement)

Vérification (doit correspondre au CSV) :
  fold 0 : 268 images (207 pucerons, 61 fonds)
  fold 1 : 370 images (250 pucerons, 120 fonds)
  fold 2 : 284 images (238 pucerons, 46 fonds)
  fold 3 : 269 images (216 pucerons, 53 fonds)
  fold 4 : 258 images (208 pucerons, 50 fonds)
  fold 5 : 318 images (228 pucerons, 90 fonds)


In [6]:
# --- CONFIG COMPARAISON (sorties + modèles + réglages) ---
SEED = 42
OUT_DIR = Path("/content/drive/MyDrive/Emma/puceron_model_2026/puceron_model_article/data")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_CV        = OUT_DIR / "comparaison_modeles_cv.csv"
CSV_RESUME    = OUT_DIR / "comparaison_modeles_resume.csv"
SAVE_DIR      = OUT_DIR / "modeles";      SAVE_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_FROZEN = OUT_DIR / "splits_figes"; SPLITS_FROZEN.mkdir(parents=True, exist_ok=True)
WANDB_PROJECT = "comparaison_pucerons_neg1"

MODELS = {"YOLO26n": "yolo26n.pt", "YOLO11n": "yolo11n.pt", "YOLO12n": "yolo12n.pt"}

COMPARE_EPOCHS = 30
BATCH          = 32
IMGSZ          = 640
NEG_RATIO      = 3
PATIENCE       = 5
PROJECT        = "yolo_comp"
CV_FOLDS       = list(range(N_CV_FOLDS))     # 0-4 ; fold5 réservé au test

# Augmentation FORCÉE (identique pour tous) ; le reste = défauts de chaque modèle
AUG = {"hsv_h":0, "hsv_s":0.2, "hsv_v":0.2, "degrees":0, "translate":0.1, "scale":0.25,
       "shear":0, "perspective":0, "flipud":0.5, "fliplr":0.5, "mosaic":1, "mixup":0.0,
       "cutmix":0, "bgr":0.0, "auto_augment":None}

def has_annotations(lbl):
    lbl = Path(lbl)
    return lbl.exists() and any(l.strip() for l in lbl.read_text().splitlines())
def _label_of(img_str):
    return Path(img_str.replace("/images/", "/labels/")).with_suffix(".txt")

print("Config comparaison OK | sorties :", OUT_DIR, "| modèles :", list(MODELS))

Config comparaison OK | sorties : /content/drive/MyDrive/Emma/puceron_model_2026/puceron_model_article/data | modèles : ['YOLO26n', 'YOLO11n', 'YOLO12n']


In [6]:
# --- Suivi W&B ---
import wandb
from ultralytics import settings
wandb.login()
settings.update({"wandb": True})
os.environ["WANDB_PROJECT"] = WANDB_PROJECT
os.environ["WANDB_TAGS"]    = "ultralytics,comparaison,cv"
print("W&B activé -> projet :", WANDB_PROJECT)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dubrule1965 (equipe) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B activé -> projet : comparaison_pucerons_neg1


In [7]:
# --- Construction du yaml par fold : liste FIGÉE sur le Drive + COPIE LOCALE lue par Ultralytics ---
# La copie locale (/content/) évite le chemin raccourci Drive (.shortcut-targets-by-id) qui plante Ultralytics.
def build_fold_yaml(fold, neg_ratio):
    root = OUTPUT_ROOT / SEARCH_VARIANT
    txt_frozen = SPLITS_FROZEN / f"train_fold{fold}_neg{neg_ratio}.txt"   # Drive : référence reproductible
    if txt_frozen.exists():
        lines = [l for l in txt_frozen.read_text().splitlines() if l.strip()]
    else:
        pos, neg = [], []
        for i in range(N_CV_FOLDS):
            if i == fold: continue
            for img in sorted((root / f"fold_{i}" / "images").glob("*")):
                lbl = root / f"fold_{i}" / "labels" / f"{img.stem}.txt"
                (pos if has_annotations(lbl) else neg).append(str(img))
        ex = root / "_extra_train"
        if (ex / "images").exists():
            for img in sorted((ex / "images").glob("*")):
                lbl = ex / "labels" / f"{img.stem}.txt"
                (pos if has_annotations(lbl) else neg).append(str(img))
        random.seed(SEED + fold)
        negR = neg if len(neg) <= neg_ratio * len(pos) else random.sample(neg, neg_ratio * len(pos))
        lines = pos + negR
        txt_frozen.write_text("\n".join(lines))                          # figé sur le Drive

    # COPIE LOCALE (chemin propre /content/) -> c'est CELLE-CI que le yaml désigne
    txt_local = YAML_DIR / f"train_fold{fold}_neg{neg_ratio}.txt"
    txt_local.write_text("\n".join(lines))

    npos = sum(1 for l in lines if has_annotations(_label_of(l)))
    nneg = len(lines) - npos
    ypath = YAML_DIR / f"compare_fold{fold}_neg{neg_ratio}.yaml"
    yaml.dump({"path": str(root), "train": str(txt_local), "val": f"fold_{fold}/images",
               "names": CLASS_NAMES}, open(ypath, "w"), sort_keys=False, allow_unicode=True)
    return ypath, npos, nneg

In [9]:
# --- COMPARAISON EN CV (métriques PAR CLASSE) — reprise auto via le CSV ---
import time, numpy as np
ncl = len(CLASS_NAMES)
CLS = [CLASS_NAMES[i] for i in range(ncl)]      # ex. ["Apterous_aphid", "Alate_aphid"]

def _by_class(values, class_index, ncl):
    """Aligne un tableau (indexé par ap_class_index) sur [classe 0 .. ncl-1]."""
    if values is None: return [0.0]*ncl
    values = [float(v) for v in list(values)]
    if len(values) == ncl: return values
    idx = list(class_index) if class_index is not None else list(range(len(values)))
    d = {int(c): v for c, v in zip(idx, values)}
    return [d.get(c, 0.0) for c in range(ncl)]

def per_class_metrics(box, ncl):
    ci = getattr(box, "ap_class_index", None)
    ap50 = _by_class(getattr(box, "ap50", None), ci, ncl)     # mAP@0.5 par classe
    ap   = _by_class(getattr(box, "maps", None), ci, ncl)     # mAP@0.5:0.95 par classe
    p    = _by_class(getattr(box, "p", None), ci, ncl)
    r    = _by_class(getattr(box, "r", None), ci, ncl)
    f1   = [2*p[i]*r[i]/(p[i]+r[i]) if (p[i]+r[i]) else 0.0 for i in range(ncl)]
    return ap50, ap, p, r, f1

def per_class_tpfpfn(ev, ncl):
    """TP/FP/FN par classe depuis la matrice de confusion (seuil conf par défaut)."""
    cm = getattr(ev, "confusion_matrix", None)
    if cm is None or getattr(cm, "matrix", None) is None:
        return [0]*ncl, [0]*ncl, [0]*ncl
    M = np.array(cm.matrix)                      # (ncl+1, ncl+1) : M[pred, gt]
    tp = [int(M[c, c]) for c in range(ncl)]
    fp = [int(M[c, :].sum() - M[c, c]) for c in range(ncl)]
    fn = [int(M[:, c].sum() - M[c, c]) for c in range(ncl)]
    return tp, fp, fn

def _epochs_info(trainer):
    last = int(getattr(trainer, "epoch", -1))
    if last >= 0: last += 1
    be = getattr(trainer, "best_epoch", None)
    if be is None or be < 0:
        st = getattr(trainer, "stopper", None)
        be = getattr(st, "best_epoch", None) if st is not None else None
    if be is not None and be >= 0: return int(be), last
    try:
        d = pd.read_csv(Path(trainer.save_dir)/"results.csv"); d.columns=[c.strip() for c in d.columns]
        col = next((c for c in d.columns if "mAP50(B)" in c), None)
        if col: return int(d[col].idxmax())+1, last
    except Exception: pass
    return -1, last

def latency_cpu_ms(weights, imgsz, warmup=10, iters=50):
    """Latence CPU par image (forward, batch=1) : moyenne ± écart-type (ms)."""
    import torch
    net = YOLO(weights).model.to("cpu").eval()
    x = torch.randn(1, 3, imgsz, imgsz)
    with torch.no_grad():
        for _ in range(warmup): net(x)
        ts = []
        for _ in range(iters):
            t0 = time.perf_counter(); net(x); ts.append((time.perf_counter()-t0)*1000)
    return float(np.mean(ts)), float(np.std(ts))

if CSV_CV.exists():
    rows = pd.read_csv(CSV_CV).to_dict("records")
    done = {(r["modele"], int(r["fold"])) for r in rows}
    print(f"Reprise : {len(done)} (modèle, fold) déjà faits.")
else:
    rows, done = [], set()

for mname, weights in MODELS.items():
    for fold in CV_FOLDS:
        if (mname, fold) in done:
            print(f"  skip {mname} fold{fold}"); continue
        try:
            yml, npos, nneg = build_fold_yaml(fold, NEG_RATIO)
            model = YOLO(weights)
            t0 = time.time()
            model.train(data=str(yml), epochs=COMPARE_EPOCHS, batch=BATCH, imgsz=IMGSZ,
                        seed=SEED, verbose=False, val=True, patience=PATIENCE,
                        project=PROJECT, name=f"{mname}_fold{fold}", exist_ok=True, **AUG)
            train_time = time.time() - t0
            be, le = _epochs_info(model.trainer)
            best_pt = Path(model.trainer.save_dir) / "weights" / "best.pt"
            if not best_pt.exists():
                raise FileNotFoundError("best.pt introuvable (entraînement interrompu ?)")
            saved = SAVE_DIR / f"{mname}_fold{fold}_best.pt"; shutil.copy2(best_pt, saved)

            ev = YOLO(str(saved)).val(data=str(yml), batch=BATCH, imgsz=IMGSZ, verbose=False)
            ap50, ap, p, r, f1 = per_class_metrics(ev.box, ncl)
            tp, fp, fn = per_class_tpfpfn(ev, ncl)
            lat_ms, lat_std = latency_cpu_ms(str(saved), IMGSZ)
            n_params = sum(pp.numel() for pp in model.model.parameters())

            row = {"modele": mname, "fold": fold,
                   "map50_macro": round(float(ev.box.map50),4), "map5095_macro": round(float(ev.box.map),4)}
            for i, nm in enumerate(CLS):
                row.update({f"{nm}_map50": round(ap50[i],4), f"{nm}_map5095": round(ap[i],4),
                            f"{nm}_P": round(p[i],4), f"{nm}_R": round(r[i],4), f"{nm}_F1": round(f1[i],4),
                            f"{nm}_TP": tp[i], f"{nm}_FP": fp[i], f"{nm}_FN": fn[i]})
            row.update({"best_epoch": be, "last_epoch": le, "train_time_s": round(train_time,1),
                        "latency_cpu_ms": round(lat_ms,3), "latency_std_ms": round(lat_std,3),
                        "n_params_M": round(n_params/1e6,3), "size_MB": round(saved.stat().st_size/1e6,2),
                        "n_pucerons_train": npos, "n_fonds_train": nneg})
            rows.append(row); pd.DataFrame(rows).to_csv(CSV_CV, index=False)
            print(f"  OK {mname} fold{fold} : map50={row['map50_macro']} | best_ep={be} fin={le} "
                  f"| latence_cpu={row['latency_cpu_ms']}ms")
        except Exception as e:
            print(f"  ERREUR {mname} fold{fold} : {type(e).__name__}: {e}"); continue

print("\nComparaison terminée ->", CSV_CV)


Ultralytics 8.4.152 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=None, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0, data=/content/yolo_yaml/yolo_neg1/compare_fold0_neg3.yaml, degrees=0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0, hsv_s=0.2, hsv_v=0.2, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1, multi_scale=0.0, name=YOLO26

Overriding model.yaml nc=80 with nc=2

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytics

wandb: WARNING Tried to log to step 10 that is less than the current step 12. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


lr/pg0,▁▅██▇▇▇▆▆▅
lr/pg1,▁▅██▇▇▇▆▆▅
lr/pg2,▁▅██▇▇▇▆▆▅
metrics/mAP50(B),▁▅▇▆█▇▇█▆▇█
metrics/mAP50-95(B),▁▅▆▆██▇█▆▇█
metrics/precision(B),▁█▅█▅▅▅▆▄▅▅
metrics/recall(B),█▁▄▃▆▅▅▅▄▄▆
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+6,...


Ultralytics 8.4.152 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
YOLO26n summary (fused): 120 layers, 2,375,226 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.2±0.0 ms, read: 112.4±37.7 MB/s, size: 31.9 KB)
val: Scanning /content/kfold_yolo/yolo_neg1/labels_cell_20/fold_0/labels.cache... 207 images, 61 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 268/268 140.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 9.4it/s 1.0s
                   all        268        362      0.529      0.534      0.512        0.3
Speed: 0.7ms preprocess, 0.8ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /content/runs/detect/val
  OK YOLO26n fold0 : map50=0.5123 | best_ep=5 fin=10 | latence_cpu=62.433ms
Ultralytics 8.4.152 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: ag

...

Ultralytics 8.4.152 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
YOLOv12n summary (fused): 159 layers, 2,557,118 parameters, 0 gradients, 7.3 GFLOPs
val: Fast image access ✅ (ping: 0.2±0.0 ms, read: 109.3±24.9 MB/s, size: 32.2 KB)
val: Scanning /content/kfold_yolo/yolo_neg1/labels_cell_20/fold_3/labels.cache... 216 images, 53 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 269/269 112.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 8.8it/s 1.0s
                   all        269        362      0.765      0.589      0.619      0.376
Speed: 0.9ms preprocess, 0.8ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to /content/runs/detect/val-14
  OK YOLO12n fold3 : map50=0.6186 | best_ep=21 fin=26 | latence_cpu=74.192ms
Ultralytics 8.4.152 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/traine

Overriding model.yaml nc=80 with nc=2

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  2    180864  ultralytics.nn.modules.block.A2C2f           [128, 128, 2, True, 4]        
  7                  -1  1    295424  ultralytics

wandb: WARNING Tried to log to step 14 that is less than the current step 16. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


lr/pg0,▁▅██▇▇▇▆▆▅▅▅▄▄
lr/pg1,▁▅██▇▇▇▆▆▅▅▅▄▄
lr/pg2,▁▅██▇▇▇▆▆▅▅▅▄▄
metrics/mAP50(B),▁▁▆▆▅▆▇▇███▇▇██
metrics/mAP50-95(B),▁▁▆▆▅▆▇▇███████
metrics/precision(B),▁▁▇██▇██▄▄▄▄▄█▄
metrics/recall(B),▁▃▆▅▃▅▆▆█▆▇▇▆▇▆
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+6,...


Ultralytics 8.4.152 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
YOLOv12n summary (fused): 159 layers, 2,557,118 parameters, 0 gradients, 7.3 GFLOPs
val: Fast image access ✅ (ping: 0.2±0.0 ms, read: 99.5±24.3 MB/s, size: 29.5 KB)
val: Scanning /content/kfold_yolo/yolo_neg1/labels_cell_20/fold_4/labels.cache... 208 images, 50 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 258/258 120.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 8.5it/s 1.1s
                   all        258        365      0.388      0.338      0.406      0.237
Speed: 0.9ms preprocess, 1.1ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to /content/runs/detect/val-15
  OK YOLO12n fold4 : map50=0.4062 | best_ep=9 fin=14 | latence_cpu=74.247ms

Comparaison terminée -> /content/drive/MyDrive/Emma/puceron_model_2026/puceron_model_article/data/comparaison_modeles_cv.csv


In [8]:
# --- EXPORT EXCEL (3 feuilles : hyperparamètres + résultats par fold + moyennes) ---
import datetime, yaml as _yaml
!pip install -q openpyxl

df = pd.read_csv(CSV_CV)
stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")
xlsx = OUT_DIR / f"comparaison_detection_{stamp}.xlsx"

# Feuille 1 : hyperparamètres réellement utilisés (args.yaml du fold0 de chaque modèle)
HP_KEYS = ["optimizer","lr0","lrf","momentum","weight_decay","warmup_epochs","warmup_momentum",
           "box","cls","dfl","batch","imgsz","epochs","patience","freeze","close_mosaic","cos_lr","nms","amp"]
hp_rows = []
for mname in MODELS:
    ap = Path(PROJECT) / f"{mname}_fold0" / "args.yaml"
    if ap.exists():
        a = _yaml.safe_load(ap.read_text())
        hp_rows.append({"date": stamp, "modele": mname, **{k: a.get(k) for k in HP_KEYS}})
    else:
        print(f"  {mname} : args.yaml introuvable ({ap})")
hp = pd.DataFrame(hp_rows) if hp_rows else pd.DataFrame([{"info": "lance d'abord la comparaison"}])

# Feuille 2 : résultats par fold
res = df.copy(); res.insert(0, "date", stamp)

# Feuille 3 : moyennes par modèle
num = [c for c in df.columns if c not in ("modele", "fold")]
moy = df.groupby("modele")[num].mean().round(4).reset_index()
moy.insert(0, "date", stamp)
moy = moy.sort_values("map50_macro", ascending=False)

with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
    hp.to_excel(w, sheet_name="hyperparametres", index=False)
    res.to_excel(w, sheet_name="resultats_par_fold", index=False)
    moy.to_excel(w, sheet_name="moyennes_par_modele", index=False)

print("Excel écrit ->", xlsx)
print("\n=== Moyennes par modèle (triées par mAP50) ===")
cols = ["modele","map50_macro","latency_cpu_ms","n_params_M","size_MB"]
print(moy[[c for c in cols if c in moy.columns]].to_string(index=False))


  YOLO26n : args.yaml introuvable (yolo_comp/YOLO26n_fold0/args.yaml)
  YOLO11n : args.yaml introuvable (yolo_comp/YOLO11n_fold0/args.yaml)
  YOLO12n : args.yaml introuvable (yolo_comp/YOLO12n_fold0/args.yaml)
Excel écrit -> /content/drive/MyDrive/Emma/puceron_model_2026/puceron_model_article/data/comparaison_detection_2026-09-15_13-31.xlsx

=== Moyennes par modèle (triées par mAP50) ===
 modele  map50_macro  latency_cpu_ms  n_params_M  size_MB
YOLO12n       0.5813         74.1020       2.568    5.520
YOLO26n       0.5676         62.9670       2.505    5.386
YOLO11n       0.5223         60.4626       2.590    5.470
